In [1]:
import pandas as pd
import networkx as nx
from pyvis.network import Network

In [2]:
# Import data into dataframe
df_city = pd.read_csv("data/sac-city-2025-04-08T19_22_35.865Z.csv")
df_county = pd.read_csv("data/sac-county-2025-04-08T19_22_35.865Z.csv")

df = pd.concat([df_city, df_county])


In [ ]:
# # NOTE: Filter out all individual donors
# df = df[df["contributorFirstName"].isna()]

# df["contributorFullName"] = pd.str.join df.contributorFirstName + " " + df.contributorLastName
df["contributorFullName"] = (
    df[["contributorFirstName", "contributorLastName"]]
      .fillna("")
      .agg(" ".join, axis=1)
      .str.strip()
)

In [ ]:
# Only include donors who donated more than the threshold
donation_threshold = 1000

df = df[df.amount > donation_threshold]
print(len(df))

212


In [5]:
# Create graph directly from the dataframe
G = nx.from_pandas_edgelist(
    df,
    source="contributorFullName",
    target="name",
    edge_attr="amount",
    create_using=nx.DiGraph()
)

In [6]:
for n in G.nodes():
    if n in df["contributorFullName"].values:
        G.nodes[n]["bipartite"] = "donor"
    if n in df["name"].values:
        G.nodes[n]["bipartite"] = "candidate"

In [7]:
# Total contributed per donor (sum of outgoing edges)
donor_totals = {
    n: sum(d["amount"] for _, _, d in G.out_edges(n, data=True))
    for n in G.nodes()
}

# Total received per candidate (sum of incoming edges)
candidate_totals = {
    n: sum(d["amount"] for _, _, d in G.in_edges(n, data=True))
    for n in G.nodes()
}

# Combine based on bipartite type
node_totals = {}
for n, d in G.nodes(data=True):
    if d.get("bipartite") == "donor":
        node_totals[n] = donor_totals[n]
    else:
        node_totals[n] = candidate_totals[n]

min_size = 5
max_size = 200

totals = list(node_totals.values())
min_total = min(totals)
max_total = max(totals)

def scale_linear(total):
    if max_total == min_total:
        return (min_size + max_size) / 2  # edge case: all equal
    return min_size + (total - min_total) / (max_total - min_total) * (max_size - min_size)

node_sizes = {n: scale_linear(total) for n, total in node_totals.items()}


In [ ]:
# PyVis visualization
net = Network(notebook=True, directed=True, height="1440px", width="100%", cdn_resources='remote', filter_menu=True)

# Add donors first
for n, d in G.nodes(data=True):
    # Who did they donate to?
    candidates_amounts = {
        candidate: value["amount"]
        for _, candidate, value # value -> {"amount", <amount>}
        in G.out_edges(n, data=True)
    }

    # Sort by amount donated
    donation_summary = "\n".join(
        f"{recipient}: ${amount:,.0f}"
        for recipient, amount
        in sorted(
            candidates_amounts.items(),
            key=lambda x: (x[1], x[0]), # Sort by donation amount, then recipient's name
            reverse=True
        )
    )

    # Donor
    if d.get("bipartite") == "donor":
        net.add_node(
            str(n),
            label=" ",
            title=f"{str(n)}\nTotal: ${node_totals[n]:,.0f}\nRecipients:\n{donation_summary}",
            color="steelblue",
            size=node_sizes[n],
            labelHighlightBold=True,
        )

# Add candidates last
for n, d in G.nodes(data=True):
    if d.get("bipartite") == "candidate":
        net.add_node(
            str(n),
            label=d.get("label", str(n)),
            title=f"{str(n)}\nTotal: ${node_totals[n]:,.0f}", # Tooltip
            color="darkorange",
            size=node_sizes[n],
            shape="circularImage",
            image=f"images/{str(n).split(" ")[1].lower()}.jpg",
            labelHighlightBold=True,
        )


for u, v, d in G.edges(data=True):
    net.add_edge(
        str(u),
        str(v),
        value=d.get("amount", 1), # Edge thickness reflects amount
        arrows="to",
        arrowStrikethrough=False,
        title=f"Contributor: {str(u)}\nRecipient: {str(v)}\nAmount: ${d["amount"]:,.0f}",
        # color="lightgray",
    )

# TODO: Combine small money donors into one bucket so they're still represented
# TODO: Unions one color, small money donors another
# TODO: On hover over donor, show the list of who they donated to and the amount
# TODO: On hover over candidate, show their top five donors ranked by amount

import json

# Define the JavaScript options for edge selection color

options = {
    "interaction": {"hover": True},
    "edges": {
        "color": {"color": "rgba(120,120,120,0.25)", "highlight": "rgba(0,0,0,0.4)", "hover": "rgba(0,0,0,0.2)"},
        "selectionWidth": 2,
        "hoverWidth": 2
    },
    "physics": {
        "enabled": True,
        "stabilization": {"iterations": 100},
        "forceAtlas2Based": {
            "gravitationalConstant": -150,
            "centralGravity": 0.005,
            "springLength": 300,
            "springConstant": 0.135,
            "damping": 0.5,
            "avoidOverlap": 0.9
        },
        "minVelocity": 0.75,
        "solver": "forceAtlas2Based",
    },
    "configure": {"enabled": True, "filter": ["physics"], "showButton": True},
}
net.set_options(json.dumps(options))
# net.toggle_physics(True)
# net.show_buttons(True)
net.show("donor_to_candidate.html")

{'Kevin McCarty': 8100.0}
Kevin McCarty: $8,100
{}

{'Kevin McCarty': 8100.0}
Kevin McCarty: $8,100
{'Kevin McCarty': 8100.0}
Kevin McCarty: $8,100
{'Kevin McCarty': 5550.0}
Kevin McCarty: $5,550
{'Kevin McCarty': 4000.0}
Kevin McCarty: $4,000
{'Kevin McCarty': 8100.0}
Kevin McCarty: $8,100
{'Kevin McCarty': 4050.0}
Kevin McCarty: $4,050
{'Kevin McCarty': 4500.0}
Kevin McCarty: $4,500
{'Kevin McCarty': 18450.0, 'Lisa Kaplan': 12000.0, 'Roger Dickinson': 6800.0, 'Karina Talamantes': 12000.0, 'Phil Pluckebaum': 6800.0, 'Caity Maple': 8600.0, 'Eric Guerra': 10700.0, 'Rick Jennings': 17100.0, 'Mai Vang': 7650.0, 'Rich Desmond': 5950.0}
Kevin McCarty: $18,450
Rick Jennings: $17,100
Lisa Kaplan: $12,000
Karina Talamantes: $12,000
Eric Guerra: $10,700
Caity Maple: $8,600
Mai Vang: $7,650
Roger Dickinson: $6,800
Phil Pluckebaum: $6,800
Rich Desmond: $5,950
{'Kevin McCarty': 5550.0}
Kevin McCarty: $5,550
{'Kevin McCarty': 3850.0}
Kevin McCarty: $3,850
{'Kevin McCarty': 4050.0}
Kevin McCarty: $4